In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from collections import defaultdict
from difflib import SequenceMatcher

import os
#Set data directory
os.chdir ("/home/kdetest/Documents/VICCO/Palynology")

In [2]:
def parse_taxon_line_simple(line):
    """
    Parse a taxon line by simple tab splitting.
    The count is the last numeric value in the line.
    """
    
    # Remove the T prefix
    if line.startswith('T'):
        line = line[1:].lstrip()
    
    # Split by tabs
    parts = line.split('\t')
    parts = [p.strip() for p in parts if p.strip() != '']
    
    taxon_id = parts[0] if len(parts) > 0 else None
    
    # Initialize values
    abundance_code = None
    present_outside = None
    count_value = None
    uncertainty = None
    additional_code = None
    
    # Go through remaining parts and classify them
    for i, part in enumerate(parts[1:], 1):  # Start from index 1 (after taxon_id)
        if part == '?':
            uncertainty = '?'
        elif part == '+':
            present_outside = '+'
        elif part in ['R', 'O', 'C', 'A', 'SA']:
            abundance_code = part
        elif part.replace('.', '', 1).isdigit():
            # This is a numeric value - it's likely the count
            count_value = part
        else:
            # Any other code goes to additional
            additional_code = part
    
    return {
        'taxon_id': taxon_id,
        'uncertainty': uncertainty,
        'abundance_code': abundance_code,
        'present_outside_sample': present_outside,
        'count_value': count_value,
        'additional_code': additional_code
    }

def parse_stratabugs_simple(file_path):
    """
    Simple parser that respects tab structure.
    """
    
    # Read file into sections
    sections = {'header': [], 'taxa': [], 'samples': [], 'abundance': []}
    
    with open(file_path, 'r', encoding='latin-1') as f:
        current_section = 'header'
        
        for line in f:
            line = line.rstrip('\n\r')
            
            if 'TAXA' in line:
                current_section = 'taxa'
                continue
            elif 'SAMPLES' in line:
                current_section = 'samples'
                continue
            elif 'ABNSCHME' in line:
                current_section = 'abundance'
                continue
            
            if line.strip():
                sections[current_section].append(line)
    
    # Parse samples section
    sample_data = []
    current_sample = None
    current_data_type = None
    current_sample_method = None
    
    for line in sections['samples']:
        if line.startswith('S'):
            # Save previous sample
            if current_sample:
                sample_data.append(current_sample.copy())
            
            # Parse sample header (space separated)
            parts = line.split()
            current_sample = {
                'depth': float(parts[1]),
                'sample_type': parts[2],
                'param1': parts[3],
                'param2': parts[4] if len(parts) > 4 else None,
                'data_type': current_data_type,
                'sample_method': current_sample_method,
                'observations': []
            }
            current_data_type = None
            current_sample_method = None
        
        elif line.startswith('D') and current_sample:
            parts = line.split()
            if len(parts) >= 2:
                current_data_type = parts[1]
                current_sample_method = parts[2] if len(parts) > 2 else None
                if current_sample:
                    current_sample['data_type'] = current_data_type
                    current_sample['sample_method'] = current_sample_method
        
        elif line.startswith('T') and current_sample:
            parsed_obs = parse_taxon_line_simple(line)
            parsed_obs['depth'] = current_sample['depth']
            parsed_obs['sample_type'] = current_sample['sample_type']
            current_sample['observations'].append(parsed_obs)
    
    # Add last sample
    if current_sample:
        sample_data.append(current_sample)
    
    # Create DataFrame from observations
    all_observations = []
    for sample in sample_data:
        for obs in sample['observations']:
            all_observations.append(obs)
    
    obs_df = pd.DataFrame(all_observations)
    
    # Parse taxa section (tab separated)
    taxa_dict = {}
    for line in sections['taxa']:
        if line.strip():
            parts = line.split('\t')
            parts = [p.strip() for p in parts if p.strip()]
            if len(parts) >= 2 and parts[0].replace('.', '', 1).isdigit():
                taxa_dict[parts[0]] = ' '.join(parts[1:]) if len(parts) > 1 else ''
    
    # Add taxon names to observations
    if not obs_df.empty:
        obs_df['taxon_name'] = obs_df['taxon_id'].map(taxa_dict)
        
        # Reorder columns
        final_cols = ['depth', 'sample_type', 'taxon_id', 'taxon_name', 'uncertainty', 
                     'abundance_code', 'present_outside_sample', 'count_value', 'additional_code']
        available_cols = [col for col in final_cols if col in obs_df.columns]
        obs_df = obs_df[available_cols].copy()
    
    return {
        'samples': sample_data,
        'observations': obs_df,
        'taxa': taxa_dict,
        'sections': sections
    }

In [3]:
def normalize_taxon_name_for_columns(taxon_name):
    """
    Normalize taxon name for use in column headers.
    Removes author names, years, qualifiers, and other non-taxonomic information.
    
    IMPROVED VERSION: Handles more edge cases
    """
    if not taxon_name or pd.isna(taxon_name):
        return "Unknown"
    
    name = str(taxon_name).strip()
    
    # Step 0: Add spaces after qualifiers that don't have them
    name = re.sub(r'(cf|aff|sp|spp)\.([a-zA-Z])', r'\1. \2', name)
    
    # Step 1: Remove s.s., s.l., sensu stricto, sensu lato, etc.
    name = re.sub(r'\s+(s\.?\s*s\.?|sensu\s+stricto|strict)', '', name, flags=re.IGNORECASE)
    name = re.sub(r'\s+(s\.?\s*l\.?|sensu\s+lato|latum)', '', name, flags=re.IGNORECASE)
    
    # Step 2: Remove qualifiers like cf., aff., sp., spp., var., subsp., etc.
    name = re.sub(r'\s+(cf|aff|sp|spp|var|subsp|form|f)\.?\b', '', name, flags=re.IGNORECASE)
    
    # Step 3: Handle forward slashes - replace with space
    name = re.sub(r'/', ' ', name)
    
    # Step 4: Remove content in parentheses (usually author names)
    name = re.sub(r'$[^)]*$', '', name)
    
    # Step 5: Remove author info - multiple patterns
    # Pattern: Author name with initials (e.g., Lentin,J. or Lentin J.)
    name = re.sub(r'\s+[A-Z][a-z]+,?\s*[A-Z]\.', '', name)
    name = re.sub(r'\s+[A-Z][a-z]+,?\s*[A-Z]\.?\s*[A-Z]\.?', '', name)
    
    # Pattern: Years with possible author parts
    name = re.sub(r',?\s*\d{4}\s*$', '', name)
    name = re.sub(r',?\s*\d{4}\s+', ' ', name)
    
    # Pattern: Standalone author initials without years
    name = re.sub(r'\s+[A-Z]\.(?:[A-Z]\.)?(?:[A-Z]\.)?', '', name)
    
    # Pattern: Author names separated by & symbol
    name = re.sub(r'\s+[A-Z][a-z]+\.?\s*&\s*[A-Z][a-z]+\.?', '', name)
    
    # Step 6: Remove trailing underscore patterns from author info
    name = re.sub(r'\s*[A-Z_]+_[A-Z_]+_\d{4}\s*$', '', name)
    
    # Step 7: Remove leftover connective words and punctuation
    name = re.sub(r'\s+&\s*$', '', name)
    name = re.sub(r'\s+[&,\.\-\_]+\s*$', '', name)
    
    # Step 8: Remove group and complex indicators
    name = re.sub(r'\s+(group|complex)\b', '', name, flags=re.IGNORECASE)
    
    # Step 9: Remove undifferentiated suffix
    name = re.sub(r'\s+undiff\.?\b', '', name, flags=re.IGNORECASE)
    
    # Step 10: Clean up multiple spaces
    name = re.sub(r'\s+', ' ', name).strip()
    
    # Replace spaces with underscores for column names
    name = name.replace(' ', '_')
    
    # Replace special characters
    name = re.sub(r'[^\w_]', '_', name)
    name = re.sub(r'_+', '_', name)
    name = re.sub(r'^_+|_+$', '', name)
    
    return name if name else "Unknown"

In [4]:
in_folder = Path("DISKOS_Paly_RAW-2")

# Collect all unique taxa across all files
all_unique_taxa = {}

for file_path in sorted(in_folder.glob("*.ASC")):
    
    try:
        # Parse the file
        parsed = parse_stratabugs_simple(str(file_path))
        
        if parsed is None or parsed['observations'].empty:
            continue
        
        # Get unique taxa from this file
        unique_taxa = parsed['observations']['taxon_name'].dropna().unique().tolist()
        
        # Normalize each taxon and add to master dictionary
        for taxon in unique_taxa:
            normalized = normalize_taxon_name_for_columns(taxon)
            if normalized != "Unknown" and len(normalized) > 3:
                if normalized not in all_unique_taxa:
                    all_unique_taxa[normalized] = normalized
                    # Store first occurrence for reference
                    all_unique_taxa[normalized + '_orig'] = taxon
        
        print(f"  Cumulative unique taxa: {len([k for k in all_unique_taxa.keys() if '_orig' not in k])}")
    
    except Exception as e:
        print(f"  ERROR processing {file_path.name}: {e}")

# Print summary and generate dictionary
print("\n" + "="*70)
print("SUMMARY AND TAXON DICTIONARY")
print("="*70)

# # Get just the normalized names (remove the _orig entries)
normalized_taxa = [k for k in all_unique_taxa.keys() if '_orig' not in k]
# print(f"\nTotal unique normalized taxa across {len(list(in_folder.glob('*.ASC')))} files: {len(normalized_taxa)}")

# print("\n" + "="*70)
# print("ALL UNIQUE TAXA WITH NORMALIZED FORMS")
# print("="*70)
# print(f"{'Original Name (50)':<50} → {'Normalized Name (30)':<30}")
# print("="*70)

# Sort and display
for normalized in sorted(normalized_taxa):
    original = all_unique_taxa.get(normalized + '_orig', 'Unknown')
    print(f"{original:<50} → {normalized:<30}")

# print("\n" + "="*70)
# print("COPY-PASTE THIS AS YOUR TAXON_NAME_DICT")
# print("="*70)
# print("TAXON_NAME_DICT = {")
# for normalized in sorted(normalized_taxa):
#     print(f"    '{normalized}': '{normalized}',")
# print("}")


  Cumulative unique taxa: 210
  Cumulative unique taxa: 362
  Cumulative unique taxa: 377
  Cumulative unique taxa: 619
  Cumulative unique taxa: 667
  Cumulative unique taxa: 710
  Cumulative unique taxa: 1303
  Cumulative unique taxa: 1441
  Cumulative unique taxa: 1463
  Cumulative unique taxa: 1576
  Cumulative unique taxa: 1638
  Cumulative unique taxa: 1668
  Cumulative unique taxa: 1674
  Cumulative unique taxa: 1685
  Cumulative unique taxa: 1718
  Cumulative unique taxa: 1735
  Cumulative unique taxa: 1749
  Cumulative unique taxa: 1758
  Cumulative unique taxa: 1806
  Cumulative unique taxa: 1819
  Cumulative unique taxa: 1822
  Cumulative unique taxa: 1916
  Cumulative unique taxa: 1960
  Cumulative unique taxa: 1991
  Cumulative unique taxa: 2011
  Cumulative unique taxa: 2041
  Cumulative unique taxa: 2145
  Cumulative unique taxa: 2169
  Cumulative unique taxa: 2546
  Cumulative unique taxa: 2546
  Cumulative unique taxa: 2585
  Cumulative unique taxa: 2637
  Cumulative u

In [19]:
def is_undifferentiated_or_spp(taxon_name):
    """
    Check if a taxon name represents an undifferentiated or spp. category.
    """
    name = str(taxon_name).lower().strip()
    
    # Check for undifferentiated indicators
    undiff_patterns = [
        r'\bspp\.?\b',
        r'\bundiff\.?\b',
        r'\bundifferentiated',
        r'\bsp\.?\s*unidentified',
    ]
    
    for pattern in undiff_patterns:
        if re.search(pattern, name):
            return True
    
    return False


def fuzzy_match_taxa(target_taxon, available_taxa, similarity_threshold=0.85):
    """
    Find all taxa that match the specified target taxon using fuzzy matching.
    
    Simpler version - just splits on spaces and compares components directly.
    """
    target_parts = str(target_taxon).strip().split()
    
    if not target_parts:
        return []
    
    target_genus = target_parts[0].lower()
    target_species = target_parts[1].lower() if len(target_parts) > 1 else None
    
    matches = []
    
    for available_taxon in available_taxa:
        avail_parts = available_taxon.strip().split()
        
        if not avail_parts:
            continue
        
        avail_genus = avail_parts[0].lower()
        avail_species = avail_parts[1].lower() if len(avail_parts) > 1 else None
        
        # Calculate similarity for each component
        genus_similarity = SequenceMatcher(None, target_genus, avail_genus).ratio()
        
        # If target is undifferentiated, ONLY match undifferentiated taxa of same genus
        if is_undifferentiated_or_spp(target_taxon):
            if not is_undifferentiated_or_spp(available_taxon):
                continue  # Skip specific species
            
            # Strict genus match for undifferentiated
            if genus_similarity >= 0.90:
                matches.append(available_taxon)
            continue
        
        # For specific species targets:
        # If target has species specified, match MUST have species too
        if target_species is None:
            # Only genus specified - match on genus alone
            if genus_similarity >= similarity_threshold:
                matches.append(available_taxon)
        else:
            # Both genus and species specified - must match on both
            if avail_species is None:
                continue  # Skip if available doesn't have species specified
            
            species_similarity = SequenceMatcher(None, target_species, avail_species).ratio()
            
            # Both must meet the threshold
            if genus_similarity >= similarity_threshold and species_similarity >= similarity_threshold:
                matches.append(available_taxon)
    
    return matches


def find_all_matches_for_targets(available_taxa, target_taxa_list, similarity_threshold=0.85):
    """
    Find all matching taxa for your target list.
    """
    print("\n" + "="*70)
    print("FINDING MATCHES FOR TARGET TAXA")
    print("="*70)
    print(f"Data contains: {len(available_taxa)} unique taxa")
    print(f"Looking for: {len(target_taxa_list)} target taxa")
    print(f"Similarity threshold: {similarity_threshold}")
    print("="*70)
    
    results = {}
    
    for target_taxon in target_taxa_list:
        matches = fuzzy_match_taxa(target_taxon, available_taxa, similarity_threshold)
        
        print(f"\nTarget: '{target_taxon}'")
        if matches:
            print(f"  ✓ Found {len(matches)} matches:")
            for match in matches:
                # Show similarity scores
                match_similarity = []
                
                target_parts = target_taxon.strip().split()
                match_parts = match.strip().split()
                
                # Compare each component
                for i, (tp, mp) in enumerate(zip(target_parts, match_parts)):
                    sim = SequenceMatcher(None, tp.lower(), mp.lower()).ratio()
                    match_similarity.append(f"{tp}/{mp}={sim:.2f}")
                
                print(f"    - {match} ({', '.join(match_similarity)})")
        else:
            print(f"  ✗ NO MATCHES")
        
        results[target_taxon] = {
            'matches': matches,
            'count': len(matches),
            'found': len(matches) > 0
        }
    
    # Summary
    found_count = sum(1 for r in results.values() if r['found'])
    print(f"\n{'='*70}")
    print(f"SUMMARY: {found_count}/{len(target_taxa_list)} targets found in data")
    print(f"{'='*70}")
    
    return results


def create_wide_format_with_target_matching(obs_data, target_taxa_list, similarity_threshold=0.85):
    """
    Create wide format DataFrame using explicit target taxa with fuzzy matching.
    """
    print(f"\n--- create_wide_format_with_target_matching called ---")
    print(f"obs_data shape: {obs_data.shape}")
    print(f"Target taxa: {len(target_taxa_list)}")
    
    if obs_data.empty:
        print("obs_data is empty -> returning empty DataFrame")
        return pd.DataFrame()
    
    # Get all unique taxa in the data
    all_taxa = obs_data['taxon_name'].dropna().unique().tolist()
    print(f"Total unique taxa in data: {len(all_taxa)}")
    
    # Build mapping from target taxa to all matching taxa in the data
    taxa_mapping = {}
    
    print(f"\nMatching target taxa to data (similarity >= {similarity_threshold}):")
    print("="*70)
    
    for target_taxon in target_taxa_list:
        matches = fuzzy_match_taxa(target_taxon, all_taxa, similarity_threshold)
        
        if matches:
            taxa_mapping[target_taxon] = matches
            print(f"\nTarget: '{target_taxon}'")
            print(f"  Matched {len(matches)} variations:")
            for match in matches:
                print(f"    - {match}")
        else:
            print(f"\nTarget: '{target_taxon}'")
            print(f"  NO MATCHES FOUND")
            taxa_mapping[target_taxon] = []
    
    # Collect all matched taxa
    all_matched_taxa = []
    for target_taxon, matches in taxa_mapping.items():
        all_matched_taxa.extend(matches)
    
    taxa_to_use = list(set(all_matched_taxa))
    
    print(f"\n{'='*70}")
    print(f"Summary:")
    print(f"  Target taxa specified: {len(target_taxa_list)}")
    print(f"  Unique taxa matched in data: {len(taxa_to_use)}")
    print(f"{'='*70}")
    
    # Get all depths
    depths = sorted(obs_data['depth'].dropna().unique())
    wide = pd.DataFrame(index=depths)
    
    # Check which value columns exist
    have_count = 'count_value' in obs_data.columns
    have_abund = 'abundance_code' in obs_data.columns
    have_present = 'present_outside_sample' in obs_data.columns
    have_uncert = 'uncertainty' in obs_data.columns
    
    print(f"\nBuilding columns...")
    
    # Build columns for each target taxon
    for target_taxon in target_taxa_list:
        matches = taxa_mapping.get(target_taxon, [])
        
        if not matches:
            continue
        
        # Get all data for this target taxon (all variations)
        subset = obs_data[obs_data['taxon_name'].isin(matches)].copy()
        
        if subset.empty:
            continue
        
        subset = subset.set_index('depth')
        
        # Create canonical column name from target taxon
        canonical_name = target_taxon.replace(' ', '_')  # Avoid issues with periods
        
        # Add count_value column
        if have_count:
            col_name = f"{canonical_name}_cnt"
            counts = subset['count_value'].apply(pd.to_numeric, errors='coerce')
            wide[col_name] = counts.groupby('depth').first().reindex(wide.index)
        
        # Add abundance_code column if desired
        if have_abund:
            col_name = f"{canonical_name}_abn"
            wide[col_name] = subset.groupby('depth')['abundance_code'].first().reindex(wide.index)
    
        # Add present_outside column if data exists
        if have_present and subset['present_outside_sample'].any():
            col_name = f"{canonical_name}_p-out"
            wide[col_name] = subset.groupby('depth')['present_outside_sample'].first().reindex(wide.index)

        # Add uncertainty column if data exists
        if have_uncert and subset['uncertainty'].any():
            col_name = f"{canonical_name}_unct"
            wide[col_name] = subset.groupby('depth')['uncertainty'].first().reindex(wide.index)
        

    print(f"\nResult shape: {wide.shape}")
    print(f"Number of columns: {len(wide.columns)}")
    
    return wide

In [17]:
"""
==========================================
# USAGE
==========================================
"""
# Taxa groups, customise as required
TARGETS = [
    'Apectodinium augustum',
    'Apectodinium cornufruticosum',  # Choose one spelling
    'Apectodinium folliculum',
    'Apectodinium homomorphum',
    'Apectodinium hyperacanthum',
    'Apectodinium paniculatum',
    'Apectodinium parvum',
    'Apectodinium quinquelatum',
    'Apectodinium summissum',
    'Apectodinium tesselatum',
    'Apectodinium undiff',
    'Spiniferites ramosus',
    'Cerodinium wardenense',
    'Deflandrea oebisfeldensis',
    'Spinidium',
    'Areoligera',
    'Azolla',
]

APECTODINIUM_SPECIES = [
    'Apectodinium augustum',
    'Apectodinium cornufruticosum',  # Choose one spelling
    'Apectodinium folliculum',
    'Apectodinium homomorphum',
    'Apectodinium hyperacanthum',
    'Apectodinium paniculatum',
    'Apectodinium parvum',
    'Apectodinium quinquelatum',
    'Apectodinium summissum',
    'Apectodinium tesselatum',
    'Apectodinium undiff',
]

SPINIFERITES_SPECIES = [
    'Spiniferites ramosus',
    # 'Spiniferites ramosus group',
]

CERODINIUM_SPECIES = [
    'Cerodinium wardenense',
]

DEFLANDREA_SPECIES = [
    'Deflandrea oebisfeldensis'
]

SPINIDIUM_SPECIES = [
    'Spinidium'
]

AREOLIGERA_SPECIES = [
    'Areoligera'
]

AZOLLA_PARTS = [
    'Azolla',
]

# Collect all taxa
in_folder = Path("DISKOS_Paly_RAW-2")
all_taxa_all_files = set()

for file_path in sorted(in_folder.glob("*.ASC")):
    try:
        parsed = parse_stratabugs_simple(str(file_path))
        if parsed and not parsed['observations'].empty:
            taxa = parsed['observations']['taxon_name'].dropna().unique().tolist()
            all_taxa_all_files.update(taxa)
    except:
        pass

print(f"\nCollected {len(all_taxa_all_files)} unique taxa across {len(list(in_folder.glob('*.ASC')))} files")

# Test undifferentiated targets
apectodinium_results = find_all_matches_for_targets(
    list(all_taxa_all_files), 
    AZOLLA_PARTS,
    similarity_threshold=0.9
)



Collected 4446 unique taxa across 78 files

FINDING MATCHES FOR TARGET TAXA
Data contains: 4446 unique taxa
Looking for: 1 target taxa
Similarity threshold: 0.9

Target: 'Azolla'
  ✓ Found 3 matches:
    - Azolla type massulae (Azolla/Azolla=1.00)
    - Azolla massulae (Azolla/Azolla=1.00)
    - Azolla spp. (Azolla/Azolla=1.00)

SUMMARY: 1/1 targets found in data


In [20]:
"""
Process files and create CSVs
"""
in_folder = Path("DISKOS_Paly_RAW-2")
out_folder = Path("DISKOS_PalyRAW2_CSV")
# out_folder = Path("PalyCSV_IndvSpCnt/Azolla_Spp")
out_folder.mkdir(exist_ok=True)

print("\n" + "="*70)
print("PROCESSING FILES")
print("="*70)

for file_path in sorted(in_folder.glob("*.ASC")):
    print(f"\nProcessing: {file_path.name}")
    
    try:
        # Parse
        parsed = parse_stratabugs_simple(str(file_path))
        
        if parsed is None or parsed['observations'].empty:
            print(f"  Skipped: No observations")
            continue
        
        # Create wide format using target matching
        wide_df = create_wide_format_with_target_matching(
            parsed['observations'],
            target_taxa_list=TARGETS,
            similarity_threshold=0.9
        )
        
        # Save if not empty
        if not wide_df.empty:
            print(f"  Columns generated: {list(wide_df.columns)}")
            
            # Save with proper depth header
            wide_df.index.name = 'depth'
            out_path = out_folder / f"{file_path.stem}.csv"
            wide_df.to_csv(out_path, index=True)
            print(f"  Saved: {out_path}")
        else:
            print(f"  Warning: Empty result")
    
    except Exception as e:
        print(f"  ERROR: {e}")
        import traceback
        traceback.print_exc()


PROCESSING FILES

Processing: 15_12-11_S_BIOSTRAT_RAW_2.ASC

--- create_wide_format_with_target_matching called ---
obs_data shape: (1697, 9)
Target taxa: 17
Total unique taxa in data: 211

Matching target taxa to data (similarity >= 0.9):

Target: 'Apectodinium augustum'
  Matched 1 variations:
    - Apectodinium augustum

Target: 'Apectodinium cornufruticosum'
  NO MATCHES FOUND

Target: 'Apectodinium folliculum'
  NO MATCHES FOUND

Target: 'Apectodinium homomorphum'
  Matched 1 variations:
    - Apectodinium homomorphum

Target: 'Apectodinium hyperacanthum'
  Matched 1 variations:
    - Apectodinium hyperacanthum

Target: 'Apectodinium paniculatum'
  NO MATCHES FOUND

Target: 'Apectodinium parvum'
  NO MATCHES FOUND

Target: 'Apectodinium quinquelatum'
  Matched 1 variations:
    - Apectodinium quinquelatum

Target: 'Apectodinium summissum'
  NO MATCHES FOUND

Target: 'Apectodinium tesselatum'
  NO MATCHES FOUND

Target: 'Apectodinium undiff'
  NO MATCHES FOUND

Target: 'Spiniferit

Traceback (most recent call last):
  File "/tmp/ipykernel_746343/1772274643.py", line 18, in <module>
    parsed = parse_stratabugs_simple(str(file_path))
  File "/tmp/ipykernel_746343/2790275078.py", line 90, in parse_stratabugs_simple
    'depth': float(parts[1]),
             ~~~~~^^^^^^^^^^
ValueError: could not convert string to float: 'Petroleum'



Target: 'Apectodinium undiff'
  NO MATCHES FOUND

Target: 'Spiniferites ramosus'
  Matched 1 variations:
    - Spiniferites ramosus group

Target: 'Cerodinium wardenense'
  NO MATCHES FOUND

Target: 'Deflandrea oebisfeldensis'
  Matched 1 variations:
    - Deflandrea oebisfeldensis

Target: 'Spinidium'
  NO MATCHES FOUND

Target: 'Areoligera'
  Matched 1 variations:
    - Areoligera spp.

Target: 'Azolla'
  NO MATCHES FOUND

Summary:
  Target taxa specified: 17
  Unique taxa matched in data: 4

Building columns...

Result shape: (9, 9)
Number of columns: 9
  Columns generated: ['Apectodinium_augustum_cnt', 'Apectodinium_augustum_abn', 'Apectodinium_augustum_p-out', 'Spiniferites_ramosus_cnt', 'Spiniferites_ramosus_abn', 'Deflandrea_oebisfeldensis_cnt', 'Deflandrea_oebisfeldensis_abn', 'Areoligera_cnt', 'Areoligera_abn']
  Saved: DISKOS_PalyRAW2_CSV/6505_10-1_BIOSTRAT_RAW_2.csv

Processing: 6507_1-1_BIOSTRAT_RAW_2.ASC
  Skipped: No observations

Processing: 6507_10-2_S_BIOSTRAT_RAW_2.A